# Verificación del entorno del curso

Este notebook comprueba, en unos dos minutos, que su computador está listo para los laboratorios de *Sistemas Inteligentes*. Revisa cuatro cosas:
1. la versión de Python y el entorno virtual del curso;
2. las librerías de los laboratorios;
3. el planificador Fast Downward, con un problema mínimo;
4. el acceso a un modelo de lenguaje, sin mostrar ninguna clave.

Cada comprobación muestra ✓ si está bien, o ✗ y qué hacer. Antes de ejecutarlo, elija en **Seleccionar kernel** (arriba a la derecha) el entorno `.venv` del curso, como explica `INSTALACION.md`.

* Ejecute las celdas en orden con **Shift + Enter**, o todas a la vez con **Ejecutar todo**.

## 1 - Python y el entorno virtual

El curso usa Python 3.13; también sirve 3.12. El notebook debe correr dentro del entorno virtual del curso, la carpeta `.venv`, para que todos los laboratorios encuentren las mismas librerías.

* Ejecute la siguiente celda para comprobar la versión de Python y el entorno.

In [ ]:
# Comprobar la versión de Python y que el kernel use un entorno virtual
import platform, sys
version = sys.version_info[:2]
print(f"{'✓' if version >= (3, 12) else '✗'} Python {version[0]}.{version[1]}"
      + ("" if version >= (3, 12) else " · el curso necesita 3.12 o superior: elija otro kernel o instale Python 3.13"))
virtual = sys.prefix != sys.base_prefix
print(f"{'✓' if virtual else '✗'} Entorno virtual: {'sí' if virtual else 'no'}"
      + ("" if virtual else " · elija el entorno .venv del curso en Seleccionar kernel"))
print(f"  Sistema operativo: {platform.system()} {platform.release()} · procesador: {platform.machine()}")

## 2 - Las librerías de los laboratorios

La celda compara las librerías instaladas con las versiones de `requirements.txt`. Si falta alguna, se instala desde VS Code: paleta de comandos, **Python: Create Environment**, o en una terminal con el entorno activo, `python -m pip install -r requirements.txt`.

* Ejecute la siguiente celda para comprobar las librerías.

In [ ]:
# Comparar las librerías instaladas con las versiones de requirements.txt
import importlib.metadata, pathlib, re
faltan = []
for linea in pathlib.Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    linea = linea.split("#")[0].strip()
    if not linea:
        continue
    paquete, _, fijada = linea.partition("==")
    try:
        instalada = importlib.metadata.version(paquete)
        ok = not fijada or instalada == fijada
        print(f"{'✓' if ok else '!'} {paquete} {instalada}" + ("" if ok else f" (el curso fija {fijada})"))
    except importlib.metadata.PackageNotFoundError:
        faltan.append(paquete)
        print(f"✗ {paquete}: no está instalado")
if faltan:
    print("\nInstale lo que falta con: python -m pip install -r requirements.txt")

## 3 - El planificador

Los laboratorios de la sesión 2 usan el planificador **Fast Downward** a través de la librería **Unified Planning**. La celda resuelve un problema mínimo, con una sola acción, y valida el plan. Si funciona, el planificador está bien instalado.

* Ejecute la siguiente celda para probar el planificador.

In [ ]:
# Resolver y validar un problema mínimo con Fast Downward
from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner, PlanValidator, get_environment
get_environment().credits_stream = None
dominio = """(define (domain prueba) (:requirements :strips)
  (:predicates (apagado) (encendido))
  (:action encender :parameters () :precondition (apagado) :effect (and (encendido) (not (apagado)))))"""
problema = """(define (problem p) (:domain prueba) (:init (apagado)) (:goal (encendido)))"""
try:
    pb = PDDLReader().parse_problem_string(dominio, problema)
    with OneshotPlanner(name="fast-downward") as planificador:
        plan = planificador.solve(pb).plan
    with PlanValidator(problem_kind=pb.kind) as validador:
        valido = validador.validate(pb, plan).status.name == "VALID"
    print(f"{'✓' if valido else '✗'} Fast Downward resolvió el problema de prueba y el plan es {'válido' if valido else 'inválido'}.")
except Exception as error:
    print(f"✗ El planificador no funcionó: {error}\n  Reinstale up-fast-downward con: python -m pip install up-fast-downward==1.0.0")

## 4 - Acceso a un modelo de lenguaje

Los laboratorios con un LLM usan la primera vía disponible: una **clave de API** guardada en el archivo `.env` de la carpeta del curso, o un **modelo local** servido con Ollama. La celda revisa las dos sin mostrar la clave: solo comprueba que exista y que tenga el formato correcto.

* Ejecute la siguiente celda para revisar el acceso a un LLM.

In [ ]:
# Revisar las vías de acceso a un LLM (no muestra ninguna clave)
import json, os, urllib.request
from dotenv import find_dotenv, load_dotenv
ruta_env = find_dotenv(usecwd=True)
if ruta_env:
    load_dotenv(ruta_env)
    print(f"✓ Archivo .env encontrado en la carpeta «{pathlib.Path(ruta_env).parent.name}».")
else:
    print("! No hay archivo .env en esta carpeta ni en las superiores.")
clave = os.environ.get("ANTHROPIC_API_KEY", "")
if not clave:
    print("! No hay ANTHROPIC_API_KEY: los laboratorios usarán el modelo local o las respuestas grabadas.")
elif clave.startswith("sk-ant-") and not any(c in clave for c in " \"'="):
    print("✓ ANTHROPIC_API_KEY tiene el formato correcto.")
else:
    print("✗ ANTHROPIC_API_KEY no tiene el formato esperado: debe ser una sola línea ANTHROPIC_API_KEY=sk-ant-..., "
          "sin espacios, sin comillas y sin un «=» de más.")
for otra in ("OPENAI_API_KEY", "GEMINI_API_KEY"):
    if os.environ.get(otra):
        print(f"✓ También hay una clave en {otra}.")
try:
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1.5) as respuesta:
        modelos = [m["name"] for m in json.load(respuesta).get("models", [])]
    tiene = "qwen3:4b-instruct" in modelos
    print(f"{'✓' if tiene else '!'} Ollama está activo" + ("" if tiene else " pero falta el modelo: ollama pull qwen3:4b-instruct"))
except Exception:
    print("· Ollama no está activo (es opcional).")

### ✏️ Prueba opcional con la API

Si ya tiene una clave, puede hacer una llamada real de prueba a Claude Haiku 5.5, el modelo por defecto del curso. Cuesta menos de una milésima de centavo de dólar.

* Cambie `PROBAR_LLM` a `True` y ejecute la celda.

In [ ]:
# ✏️ Prueba opcional: una llamada real a la API (cambie PROBAR_LLM a True)
PROBAR_LLM = False
if PROBAR_LLM:
    import anthropic
    cliente = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "").strip())
    r = cliente.messages.create(model="claude-haiku-5-5", max_tokens=20, thinking={"type": "disabled"},
                                messages=[{"role": "user", "content": "Responde solo con la palabra: listo"}])
    print("✓ Respuesta de la API:", "".join(getattr(b, "text", "") or "" for b in r.content))
else:
    print("· Prueba con la API omitida (PROBAR_LLM = False).")

## Resultado

Si las comprobaciones 1 a 3 muestran ✓, su entorno está listo para los laboratorios. Para los que usan un LLM, basta con una de las dos vías de la sección 4; sin ninguna, esos laboratorios reproducen respuestas grabadas y corren completos, aunque no pueden probar instrucciones nuevas.

Si algo muestra ✗, siga la indicación del mensaje o consulte la tabla de problemas frecuentes de `INSTALACION.md`.